# Dallas TL Attribution Run

This notebook previews the next-token prediction for:

```text
Fact: the capital of the state containing Dallas is
```

Then it runs the TransformerLens-backed attribution path through `BiologyAttributionRunner`, which imports `biology_server_t_lens` for the replacement-model forward and attribution pass. It saves the resulting attribution graph JSON plus an optional compact `.pt` summary.

## 1. Repo And Dependency Setup

Run this in a GPU Colab runtime. A high-memory A100 is the intended target for the default `BATCH_SIZE=128` / `MAX_FEATURE_NODES=300` settings.

In [1]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

# 1. Define repository remote URL, target branch, and the local clone destination
REPO_URL = "https://github.com/rowan-dauria/llm-biology.git"
BRANCH = "bio-serv-transformer-lens"
DEFAULT_COLAB_REPO = Path("/content/llm-biology")

# 2. Always clean up any existing repository folder to guarantee a fresh download
if DEFAULT_COLAB_REPO.exists():
    print(f"[SETUP] Removing existing folder {DEFAULT_COLAB_REPO} to ensure a fresh clone...")
    try:
        shutil.rmtree(DEFAULT_COLAB_REPO)
    except Exception as e:
        raise RuntimeError(
            f"[SETUP ERROR] Failed to clean up existing folder '{DEFAULT_COLAB_REPO}': {e}. "
            "Please manually delete the folder or restart your Jupyter kernel."
        ) from e

# 3. Run git clone and log progress loudly; fail loudly with instructions if it fails
print(f"[SETUP] Cloning branch '{BRANCH}' from '{REPO_URL}' to '{DEFAULT_COLAB_REPO}'...")
try:
    subprocess.check_call(["git", "clone", "--branch", BRANCH, REPO_URL, str(DEFAULT_COLAB_REPO)])
    print("[SETUP] Git clone completed successfully!")
except Exception as e:
    raise RuntimeError(
        f"[SETUP ERROR] Git clone failed for '{REPO_URL}' on branch '{BRANCH}'.\n"
        "Verify that your network connection is active and that the branch has been pushed to GitHub."
    ) from e

# 4. Point working directory and python sys.path search to the freshly cloned repo
repo_dir = DEFAULT_COLAB_REPO
os.chdir(repo_dir)
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

# 5. Add local circuit-tracer reference to search path if present locally in parent directory
local_circuit_tracer = repo_dir.parent / "circuit-tracer"
if local_circuit_tracer.exists() and str(local_circuit_tracer) not in sys.path:
    sys.path.insert(0, str(local_circuit_tracer))

print(f"[SETUP] Using active repo directory: {repo_dir}")
print(f"[SETUP] Python executable: {sys.executable}")

[SETUP] Cloning branch 'bio-serv-transformer-lens' from 'https://github.com/rowan-dauria/llm-biology.git' to '/content/llm-biology'...
[SETUP] Git clone completed successfully!
[SETUP] Using active repo directory: /content/llm-biology
[SETUP] Python executable: /usr/bin/python3


In [2]:
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
INSTALL_DEPS = IN_COLAB

if INSTALL_DEPS:
    packages = [
        "accelerate",
        "circuit-tracer",
        "einops",
        "fsspec",
        "huggingface-hub<1.0",
        "safetensors",
        "sentencepiece",
        "transformer-lens>=2.16.0",
        "transformers>=4.56.0,<=4.57.3",
        "tqdm",
        "zstandard",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])
else:
    print("Skipping dependency install outside Colab. Set INSTALL_DEPS=True to force it.")

KeyboardInterrupt: 

In [ ]:
# Keep this notebook compatible with both circuit-tracer factory names.
import importlib
import sys

slt_module = importlib.import_module("circuit_tracer.transcoder.single_layer_transcoder")
if not hasattr(slt_module, "load_transcoder"):
    if not hasattr(slt_module, "load_relu_transcoder"):
        raise ImportError(
            "circuit_tracer.transcoder.single_layer_transcoder has neither "
            "load_transcoder nor load_relu_transcoder"
        )
    slt_module.load_transcoder = slt_module.load_relu_transcoder
    print("Added load_transcoder alias for this notebook runtime.")
else:
    print("circuit-tracer already exposes load_transcoder.")

# Drop any half-imported modules left by a failed earlier run in this kernel.
for module_name in list(sys.modules):
    if module_name == "biology_server" or module_name.startswith("biology_server."):
        del sys.modules[module_name]

circuit-tracer already exposes load_transcoder.


In [ ]:
# Optional: use a Colab secret named HF_TOKEN if you need authenticated HF downloads.
try:
    from google.colab import userdata
    from huggingface_hub import login

    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        login(token=hf_token)
        print("Logged in to Hugging Face with HF_TOKEN.")
    else:
        print("No HF_TOKEN Colab secret found; continuing unauthenticated.")
except Exception as exc:
    print(f"HF login skipped: {exc}")

HF login skipped: No module named 'google.colab'


In [ ]:
import torch

print(f"torch={torch.__version__}")
print(f"cuda_available={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"gpu={torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"gpu_memory_gb={props.total_memory / 1024**3:.1f}")

torch=2.12.0
cuda_available=False


## 2. Configure The Run

`USE_CHAT_TEMPLATE=False` is intentional for this factual completion prompt. The preview target token is passed into graph generation so the attribution run targets exactly the token shown in the preview.

In [ ]:
import inspect
from datetime import datetime
from pathlib import Path

import biology_server.attribution as attribution_module
import biology_server_t_lens
from biology_server.attribution import DEFAULT_LAYERS, BiologyAttributionRunner

assert "biology_server_t_lens" in inspect.getsource(BiologyAttributionRunner._ensure_loaded)
assert "biology_server_t_lens" in inspect.getsource(BiologyAttributionRunner.generate_graph)
print(f"Using TL backend package: {biology_server_t_lens.__file__}")

PROMPT = "Fact: the capital of the state containing Dallas is"
SLUG = f"{datetime.now().strftime('%Y-%m-%d-%H-%M-%S')}-dallas-state-capital-tl"
USE_CHAT_TEMPLATE = False

LAYERS = DEFAULT_LAYERS
MAX_FEATURE_NODES = 3000
BATCH_SIZE = 32

OUTPUT_DIR = Path("data/colab_attribution_graphs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SAVE_PT_PATH = OUTPUT_DIR / f"{SLUG}.pt"

# Feature-example sidecars need data/feature_topk/150k-pile. They are not needed
# for the raw graph artifact, so skip them by default in Colab.
SKIP_FEATURE_EXAMPLES = True
if SKIP_FEATURE_EXAMPLES:

    def _skip_feature_examples(**_kwargs):
        print("[INFO] skipping feature-example sidecars")
        return {}

    attribution_module.build_feature_examples = _skip_feature_examples

runner = BiologyAttributionRunner(
    layers=LAYERS,
    graph_file_dir=OUTPUT_DIR,
    batch_size=BATCH_SIZE,
    max_feature_nodes=MAX_FEATURE_NODES,
)

print(f"prompt={PROMPT!r}")
print(f"slug={SLUG}")
print(f"layers={LAYERS}")
print(f"output_dir={OUTPUT_DIR.resolve()}")

Using TL backend package: /Users/rowandauria/Documents/GitHub/mphil-project/llm-biology/biology_server_t_lens/__init__.py
prompt='Fact: the capital of the state containing Dallas is'
slug=2026-05-30-20-54-49-dallas-state-capital-tl
layers=[2, 12, 24, 33]
output_dir=/Users/rowandauria/Documents/GitHub/mphil-project/llm-biology/data/colab_attribution_graphs


## 3. Preview The Logit Prediction

In [ ]:
preview = runner.preview(
    PROMPT,
    slug=SLUG,
    use_chat_template=USE_CHAT_TEMPLATE,
)

print("Prompt tokens:")
for idx, token in enumerate(preview.prompt_tokens):
    print(f"  {idx:02d}: {token!r}")

print("\nTarget token:")
print(
    f"  id={preview.target_token_id} "
    f"text={preview.target_token_str!r} "
    f"prob={preview.target_token_prob:.6f}"
)

print("\nTop preview tokens:")
for item in preview.top_tokens:
    print(f"  id={item.token_id:<8} p={item.prob:.6f} text={item.token!r}")

[INFO] device=mps dtype=torch.float16 layers=[2, 12, 24, 33]


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


[START] Loading preview model


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[DONE]  Loading preview model (9.8s)
[INFO] prompt tokens (10): ['Fact', ':', ' the', ' capital', ' of', ' the', ' state', ' containing', ' Dallas', ' is']
[START] Preview inference forward pass
[INFO] preview target token id=19260 (' Austin') p=0.9322
[DONE]  Preview inference forward pass (0.7s)
Prompt tokens:
  00: 'Fact'
  01: ':'
  02: ' the'
  03: ' capital'
  04: ' of'
  05: ' the'
  06: ' state'
  07: ' containing'
  08: ' Dallas'
  09: ' is'

Target token:
  id=19260 text=' Austin' prob=0.932246

Top preview tokens:
  id=19260    p=0.932246 text=' Austin'
  id=11002    p=0.026446 text=' Fort'
  id=220      p=0.006845 text=' '
  id=18542    p=0.004892 text=' Dallas'
  id=279      p=0.004632 text=' the'
  id=8257     p=0.004489 text=' Texas'
  id=7407     p=0.002039 text=' located'
  id=2598     p=0.001717 text=' called'
  id=55210    p=0.001266 text=' Irving'
  id=198      p=0.001100 text='\n'


## 4. Run TransformerLens Attribution And Save The Graph

In [ ]:
result = runner.generate_graph(
    PROMPT,
    slug=SLUG,
    target_token_id=preview.target_token_id,
    preview_top_token_id=preview.target_token_id,
    preview_top_token=preview.target_token_str,
    preview_top_token_prob=preview.target_token_prob,
    save_pt=str(SAVE_PT_PATH),
    use_chat_template=USE_CHAT_TEMPLATE,
)

print("Saved attribution graph:")
print(f"  graph_json={result.graph_path}")
print(f"  compact_pt={result.pt_path}")
print(f"  target={result.target_token_str!r} p={result.target_token_prob:.6f}")
print(f"  feature_nodes={len(result.selected_features)}")
print(f"  links={len(result.links)}")
print(f"  logit_targets={len(result.logit_targets)}")

[INFO] device=mps dtype=torch.float16 layers=[2, 12, 24, 33]


/Users/rowandauria/miniconda3/envs/qwen-sae-mac/lib/python3.12/site-packages/transformer_lens/config/HookedTransformerConfig.py:354: UserWarning: MPS backend may produce silently incorrect results (PyTorch 2.12.0). Set TRANSFORMERLENS_ALLOW_MPS=1 to suppress this warning. See: https://github.com/TransformerLensOrg/TransformerLens/issues/1178
  warn_if_mps(self.device)


[START] Loading model


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loaded pretrained model Qwen/Qwen3-4B into HookedTransformer
[DONE]  Loading model (153.3s)
[INFO] prompt tokens (10): ['Fact', ':', ' the', ' capital', ' of', ' the', ' state', ' containing', ' Dallas', ' is']
[START] TL base forward parity check
[INFO] preview/TL top-token parity ok: id=19260 (' Austin') preview_p=0.9322 tl_p=0.9320
[INFO] primary logit id=19260 (' Austin') p=0.9320 logit=20.312; logit nodes=1
[DONE]  TL base forward parity check (1.0s)
[INFO] device=mps dtype=torch.float16 layers=[2, 12, 24, 33]
[START] Resolving transcoder paths


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

[DONE]  Resolving transcoder paths (0.3s)
[START] Loading transcoder layer 2
  d_model=2560  d_transcoder=163840  skip=False
[DONE]  Loading transcoder layer 2 (1.5s)
[START] Loading transcoder layer 12
  d_model=2560  d_transcoder=163840  skip=False
[DONE]  Loading transcoder layer 12 (2.6s)
[START] Loading transcoder layer 24
  d_model=2560  d_transcoder=163840  skip=False
[DONE]  Loading transcoder layer 24 (2.2s)
[START] Loading transcoder layer 33
  d_model=2560  d_transcoder=163840  skip=False
[DONE]  Loading transcoder layer 33 (3.0s)
[INFO] loaded 1128 feature labels
[START] TL replacement forward pass
[INFO] active features=7694
[DONE]  TL replacement forward pass (33.5s)
[START] TL iterative top-K attribution
[INFO] selected feature rows=3000 unpruned links=292275
[DONE]  TL iterative top-K attribution (23.2s)
[START] Anthropic-style graph pruning
[INFO] pruned graph features=218/3000 links=9758/292275
[DONE]  Anthropic-style graph pruning (0.2s)
[START] Per-feature direct-lo

## 5. Inspect And Download Saved Artifacts

In [ ]:
import json

graph_path = Path(result.graph_path)
with graph_path.open(encoding="utf-8") as handle:
    graph = json.load(handle)

print(json.dumps(graph["metadata"], indent=2))
print(f"nodes={len(graph['nodes'])}")
print(f"links={len(graph['links'])}")
print(f"json_size_mb={graph_path.stat().st_size / 1024**2:.2f}")
if result.pt_path is not None:
    pt_path = Path(result.pt_path)
    print(f"pt_size_mb={pt_path.stat().st_size / 1024**2:.2f}")

{
  "slug": "2026-05-30-20-54-49-dallas-state-capital-tl",
  "scan": "./data/features/qwen3-4b-transcoders",
  "transcoder_list": [],
  "prompt_tokens": [
    "Fact",
    ":",
    " the",
    " capital",
    " of",
    " the",
    " state",
    " containing",
    " Dallas",
    " is"
  ],
  "prompt": "Fact: the capital of the state containing Dallas is",
  "node_threshold": 0.8,
  "schema_version": 1
}
nodes=264
links=9758
json_size_mb=1.09
pt_size_mb=0.41


In [ ]:
# Optional Colab download. The files are already saved under OUTPUT_DIR.
DOWNLOAD_FROM_COLAB = False

if DOWNLOAD_FROM_COLAB:
    from google.colab import files

    files.download(str(result.graph_path))
    if result.pt_path is not None:
        files.download(str(result.pt_path))

In [ ]:
import shutil
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

src_dir = Path("/content/llm-biology/data/colab_attribution_graphs")
dst_dir = Path("/content/drive/MyDrive/mphil-project/colab_attribution_graphs")
dst_dir.mkdir(parents=True, exist_ok=True)

for name in ["dallas-state-capital-tl.json", "dallas-state-capital-tl.pt"]:
    src = src_dir / name
    dst = dst_dir / name
    shutil.copy2(src, dst)
    print(dst, dst.stat().st_size)

ModuleNotFoundError: No module named 'google.colab'